# SRΨ-v1.0 Baseline Training (Colab - Standalone)

**版本**: v1.0-Standalone

**特点**:
- ✅ 完全自包含，不需要 git clone
- ✅ 所有代码直接在 notebook 中创建
- ✅ 不会有模块导入缓存问题
- ✅ 开箱即用

**预期性能**: Energy Drift ~10.88

---

## Phase 1: 环境准备

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 安装依赖
!pip install torch torchvision numpy pyyaml tqdm tensorboard -q

import torch
import numpy as np
print(f"\n✅ PyTorch 版本: {torch.__version__}")
print(f"✅ CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 创建项目目录结构
import os

WORK_DIR = '/content/srpsi_v1_baseline'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/src', exist_ok=True)
os.makedirs(f'{WORK_DIR}/src/models', exist_ok=True)
os.makedirs(f'{WORK_DIR}/config', exist_ok=True)

os.chdir(WORK_DIR)
import sys
sys.path.insert(0, WORK_DIR)

print(f"✅ 项目目录已创建: {WORK_DIR}")

In [ ]:
# 创建必要的数据文件
%%writefile src/data_gen.py
import numpy as np

def generate_burgers_1d(num_samples=1000, total_steps=100, nx=128, 
                        dt=0.01, dx=0.01, nu=0.01, seed=42):
    """Generate 1D Burgers equation data.
    
    Args:
        num_samples: Number of trajectories
        total_steps: Total time steps per trajectory
        nx: Spatial resolution
        dt: Time step size
        dx: Spatial step size
        nu: Viscosity coefficient
        seed: Random seed
    
    Returns:
        data: [num_samples, total_steps, nx]
    """
    np.random.seed(seed)
    
    # Domain
    x = np.linspace(0, 2*np.pi, nx, endpoint=False)
    
    data = []
    for _ in range(num_samples):
        # Random initial condition
        A = np.random.uniform(0.5, 1.5)
        k = np.random.uniform(1, 4)
        u = A * np.sin(k * x) + 0.1 * np.random.randn(nx)
        
        # Time integration
        trajectory = [u.copy()]
        for _ in range(total_steps - 1):
            # Simple finite difference
            u_xx = np.roll(u, -1) - 2*u + np.roll(u, 1)
            u = u - dt * u * np.roll(u, -1) - np.roll(u, 1) / (2*dx) + nu * u_xx / (dx**2)
            trajectory.append(u.copy())
        
        data.append(np.array(trajectory))
    
    return np.array(data)

print("✅ src/data_gen.py 已创建")

In [ ]:
# 创建数据加载文件
%%writefile src/datasets.py
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class FieldRolloutDataset(Dataset):
    def __init__(self, array, tin, tout):
        self.array = array
        self.tin = tin
        self.tout = tout
    
    def __len__(self):
        return len(self.array)
    
    def __getitem__(self, idx):
        seq = self.array[idx]
        x_in = seq[:self.tin]
        y_out = seq[self.tin:self.tin + self.tout]
        return {
            'x': torch.tensor(x_in, dtype=torch.float32),
            'y': torch.tensor(y_out, dtype=torch.float32)
        }

def create_dataloaders(data_path, tin, tout, batch_size, 
                        num_train, num_val, num_test,
                        num_workers=0, seed=42):
    data = np.load(data_path)
    num_samples = len(data)
    
    # Split
    train_data = data[:num_train]
    val_data = data[num_train:num_train+num_val]
    test_data = data[num_train+num_val:num_train+num_val+num_test]
    
    train_dataset = FieldRolloutDataset(train_data, tin, tout)
    val_dataset = FieldRolloutDataset(val_data, tin, tout)
    test_dataset = FieldRolloutDataset(test_data, tin, tout)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                              shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                             shuffle=False, num_workers=num_workers)
    
    return train_loader, val_loader, test_loader

print("✅ src/datasets.py 已创建")

In [ ]:
# 创建模型文件
%%writefile src/models/srpsi_engine_tiny.py
import torch
import torch.nn as nn

def _init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight, gain=0.01)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)
    elif isinstance(module, nn.Conv1d):
        nn.init.xavier_uniform_(module.weight, gain=0.01)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)

class InputEncoder(nn.Module):
    def __init__(self, tin, nx, hidden_dim):
        super().__init__()
        self.tin = tin
        self.nx = nx
        self.hidden_dim = hidden_dim
        
        self.net = nn.Sequential(
            nn.Linear(tin, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
        )
        
        self.apply(_init_weights)
    
    def forward(self, x):
        # x: [B, Tin, X]
        # Normalize (关键!)
        x_mean = x.mean(dim=1, keepdim=True)
        x_std = x.std(dim=1, keepdim=True) + 1e-6
        x = (x - x_mean) / x_std
        
        x = x.transpose(1, 2)  # [B, X, Tin]
        z = self.net(x)
        z = torch.clamp(z, -10, 10)
        
        real, imag = torch.chunk(z, 2, dim=-1)
        psi = torch.stack([real, imag], dim=-1)  # [B, X, D, 2]
        return psi

class StructureOperatorS(nn.Module):
    def __init__(self, hidden_dim, kernel_size=5):
        super().__init__()
        padding = kernel_size // 2
        self.real_conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.imag_conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.mix = nn.Linear(hidden_dim, hidden_dim)
        self.apply(_init_weights)
    
    def forward(self, psi):
        real = psi[..., 0].transpose(1, 2)
        imag = psi[..., 1].transpose(1, 2)
        
        real_out = self.real_conv(real).transpose(1, 2)
        imag_out = self.imag_conv(imag).transpose(1, 2)
        
        real_out = self.mix(real_out)
        imag_out = self.mix(imag_out)
        
        return torch.stack([real_out, imag_out], dim=-1)

class RhythmOperatorR(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.phase = nn.Parameter(torch.zeros(hidden_dim))
    
    def forward(self, psi):
        real = psi[..., 0]
        imag = psi[..., 1]
        
        cos_p = torch.cos(self.phase)
        sin_p = torch.sin(self.phase)
        
        real_out = real * cos_p - imag * sin_p
        imag_out = real * sin_p + imag * cos_p
        
        return torch.stack([real_out, imag_out], dim=-1)

class StableProjection(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim * 2)
        )
        self.apply(_init_weights)
    
    def forward(self, psi):
        B, X, D, _ = psi.shape
        psi_flat = psi.view(B, X, -1)
        out = self.proj(psi_flat)
        out = out.view(B, X, D, 2)
        return out

class SRPsiBlock(nn.Module):
    def __init__(self, hidden_dim, kernel_size=5):
        super().__init__()
        self.S = StructureOperatorS(hidden_dim, kernel_size)
        self.R = RhythmOperatorR(hidden_dim)
        self.N = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.GELU(),
        )
        self.Phi = StableProjection(hidden_dim)
        self.apply(_init_weights)
    
    def forward(self, psi):
        dpsi_S = self.S(psi)
        dpsi_R = self.R(psi)
        
        psi_S = psi + dpsi_S
        psi_R = psi_S + dpsi_R
        
        psi_flat = torch.cat([psi_R[..., 0], psi_R[..., 1]], dim=-1)
        modulation = self.N(psi_flat)
        dpsi_N = torch.stack(modulation.chunk(2, dim=-1), dim=-1)
        
        psi_new = psi_R + dpsi_N
        psi_out = self.Phi(psi_new)
        
        return psi_out

class OutputDecoder(nn.Module):
    def __init__(self, hidden_dim, nx, tout):
        super().__init__()
        self.tout = tout
        self.nx = nx
        self.hidden_dim = hidden_dim
        
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, tout * nx)
        )
        self.apply(_init_weights)
    
    def forward(self, psi):
        B, X, D, _ = psi.shape
        psi_flat = psi.view(B, X, -1)
        out = self.decoder(psi_flat)
        out = out.view(B, self.nx, self.tout)
        out = out.transpose(1, 2)  # [B, Tout, X]
        return out

class SRPsiEngineTiny(nn.Module):
    def __init__(self, tin, nx, hidden_dim=64, depth=3, 
                 kernel_size=5, dt=0.01, tout=32):
        super().__init__()
        self.tin = tin
        self.nx = nx
        self.hidden_dim = hidden_dim
        self.tout = tout
        
        self.encoder = InputEncoder(tin, nx, hidden_dim)
        self.blocks = nn.ModuleList([
            SRPsiBlock(hidden_dim, kernel_size) for _ in range(depth)
        ])
        self.decoder = OutputDecoder(hidden_dim, nx, tout)
    
    def forward(self, x):
        # x: [B, Tin, X]
        psi = self.encoder(x)  # [B, X, D, 2]
        
        for block in self.blocks:
            psi = block(psi)
        
        out = self.decoder(psi)  # [B, Tout, X]
        return out

print("✅ src/models/srpsi_engine_tiny.py 已创建")

In [ ]:
# 创建模型初始化文件
%%writefile src/models/__init__.py
from .srpsi_engine_tiny import SRPsiEngineTiny

__all__ = ['SRPsiEngineTiny']
print("✅ src/models/__init__.py 已创建")

In [ ]:
# 创建工具函数
%%writefile src/utils.py
import torch
import yaml
import random
import numpy as np
from pathlib import Path

def load_config(config_path):
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device(device_str):
    if device_str == 'cuda' and torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

def create_output_dir(output_dir, model_name):
    output_dir = Path(output_dir) / model_name
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = output_dir / 'checkpoints'
    checkpoint_dir.mkdir(exist_ok=True)
    return output_dir

def save_checkpoint(model, optimizer, epoch, val_loss, path, **kwargs):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
        **kwargs
    }
    torch.save(checkpoint, path)

class AverageMeter:
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print("✅ src/utils.py 已创建")

In [ ]:
# 创建损失函数
%%writefile src/losses.py
import torch

def total_loss(model, x, pred, y, cfg, epoch=0, compute_shift=False):
    # Prediction loss
    loss_pred = torch.mean((pred - y) ** 2)
    
    # Conservation loss (energy)
    energy_pred = 0.5 * torch.sum(pred ** 2, dim=[1, 2])
    energy_target = 0.5 * torch.sum(y ** 2, dim=[1, 2])
    loss_cons = torch.mean(torch.abs(energy_pred - energy_target))
    
    # Total loss
    lambda_cons = cfg['loss'].get('lambda_cons', 0.1)
    loss_total = loss_pred + lambda_cons * loss_cons
    
    logs = {
        'loss_total': loss_total.item(),
        'loss_pred': loss_pred.item(),
        'loss_cons': loss_cons.item(),
        'loss_phase': 0.0,
        'loss_smooth': 0.0
    }
    
    return loss_total, logs

print("✅ src/losses.py 已创建")

In [ ]:
# 创建评估指标
%%writefile src/metrics.py
import torch

def rollout_mse(pred, target):
    return torch.mean((pred - target) ** 2).item()

def energy_drift(pred, target):
    energy_pred = 0.5 * torch.sum(pred ** 2, dim=[1, 2])
    energy_target = 0.5 * torch.sum(target ** 2, dim=[1, 2])
    drift = torch.abs(energy_pred - energy_target).mean()
    return drift.item()

print("✅ src/metrics.py 已创建")

In [ ]:
# 创建配置文件
%%writefile config/default.yaml
seed: 42
device: cuda

task:
  name: burgers_1d
  nx: 128
  tin: 16
  tout: 32
  dt: 0.01
  dx: 0.01
  samples_train: 4000
  samples_val: 400
  samples_test: 400
  noise_std: 0.00

training:
  batch_size: 32
  epochs: 80
  lr: 0.0001
  weight_decay: 1.0e-5
  grad_clip: 0.5

loss:
  lambda_cons: 0.1
  lambda_phase: 0.0
  lambda_smooth: 0.02

model:
  hidden_dim: 64
  depth: 3
  kernel_size: 5
  dropout: 0.0

output:
  log_interval: 10
  save_interval: 20
print("✅ config/default.yaml 已创建")

In [ ]:
# 创建任务配置文件
%%writefile config/burgers.yaml
inherit: default.yaml

task:
  name: burgers_1d
  nu: 0.01
  domain_size: 2 * pi

data_gen:
  ic_type: random
  amp_range: [0.5, 1.5]
  freq_range: [1, 4]
  bump_prob: 0.5
print("✅ config/burgers.yaml 已创建")

In [ ]:
# 创建训练脚本
%%writefile train.py
import sys
import os
sys.path.insert(0, os.getcwd())

import argparse
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from tqdm import tqdm

from src.utils import load_config, set_seed, get_device, create_output_dir, save_checkpoint, count_parameters
from src.datasets import create_dataloaders
from src.models.srpsi_engine_tiny import SRPsiEngineTiny
from src.losses import total_loss
from src.metrics import rollout_mse, energy_drift

def create_model(cfg, device):
    model = SRPsiEngineTiny(
        tin=cfg['task']['tin'],
        nx=cfg['task']['nx'],
        hidden_dim=cfg['model']['hidden_dim'],
        depth=cfg['model']['depth'],
        kernel_size=cfg['model']['kernel_size'],
        dt=cfg['task']['dt'],
        tout=cfg['task']['tout']
    )
    return model.to(device)

def train_epoch(model, dataloader, optimizer, cfg, device, epoch):
    model.train()
    
    from src.utils import AverageMeter
    loss_meter = AverageMeter()
    pred_meter = AverageMeter()
    cons_meter = AverageMeter()
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
    
    for batch in pbar:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        
        pred = model(x)
        loss, logs = total_loss(model, x, pred, y, cfg, epoch)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['training']['grad_clip'])
        optimizer.step()
        
        loss_meter.update(logs["loss_total"], x.shape[0])
        pred_meter.update(logs["loss_pred"], x.shape[0])
        cons_meter.update(logs["loss_cons"], x.shape[0])
        
        pbar.set_postfix({"loss": f"{loss_meter.avg:.6f}"})
    
    return {"train_loss": loss_meter.avg}

@torch.no_grad()
def validate(model, dataloader, cfg, device):
    model.eval()
    
    loss_meter = 0.0
    rollout_meter = 0.0
    drift_meter = 0.0
    
    for batch in dataloader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        
        pred = model(x)
        loss, _ = total_loss(model, x, pred, y, cfg)
        
        loss_meter += loss.item() * x.shape[0]
        rollout_meter += rollout_mse(pred, y) * x.shape[0]
        drift_meter += energy_drift(pred, y) * x.shape[0]
    
    n = len(dataloader.dataset)
    return {
        "val_loss": loss_meter / n,
        "val_rollout_mse": rollout_meter / n,
        "val_energy_drift": drift_meter / n,
    }

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, default="config/burgers.yaml")
    parser.add_argument("--model", type=str, default="srpsi_engine")
    parser.add_argument("--data", type=str, default=None)
    parser.add_argument("--epochs", type=int, default=None)
    parser.add_argument("--output", type=str, default="outputs/experiment")
    args = parser.parse_args()
    
    cfg = load_config(args.config)
    if args.epochs is not None:
        cfg["training"]["epochs"] = args.epochs
    
    set_seed(cfg["seed"])
    device = get_device(cfg.get("device", "cuda"))
    
    output_dir = create_output_dir(args.output, args.model)
    print(f"Output directory: {output_dir}")
    
    # Data
    if args.data is None:
        print("请提供数据路径: --data data/burgers_1d.npy")
        return
    
    train_loader, val_loader, test_loader = create_dataloaders(
        data_path=args.data,
        tin=cfg['task']['tin'],
        tout=cfg['task']['tout'],
        batch_size=cfg['training']['batch_size'],
        num_train=cfg['task']['samples_train'],
        num_val=cfg['task']['samples_val'],
        num_test=cfg['task']['samples_test'],
        num_workers=0
    )
    
    # Model
    model = create_model(cfg, device)
    print(f"Model parameters: {count_parameters(model):,}")
    
    optimizer = optim.Adam(
        model.parameters(),
        lr=cfg['training']['lr'],
        weight_decay=cfg['training']['weight_decay']
    )
    
    # Training
    print(f"\nStarting training for {cfg['training']['epochs']} epochs...")
    best_val_loss = float('inf')
    
    for epoch in range(cfg['training']['epochs']):
        train_metrics = train_epoch(model, train_loader, optimizer, cfg, device, epoch)
        val_metrics = validate(model, val_loader, cfg, device)
        
        print(f"Epoch {epoch+1}/{cfg['training']['epochs']}")
        print(f"  Train Loss: {train_metrics['train_loss']:.6f}")
        print(f"  Val Loss:   {val_metrics['val_loss']:.6f}")
        print(f"  Val Drift:  {val_metrics['val_energy_drift']:.4f}")
        
        if val_metrics['val_loss'] < best_val_loss:
            best_val_loss = val_metrics['val_loss']
            save_checkpoint(
                model, optimizer, epoch, val_metrics['val_loss'],
                str(output_dir / 'checkpoints' / 'best.pt'),
                best_val_loss=best_val_loss
            )
            print(f"  ✓ New best model: {best_val_loss:.6f}")
        
        if (epoch + 1) % cfg['output']['save_interval'] == 0:
            save_checkpoint(
                model, optimizer, epoch, val_metrics['val_loss'],
                str(output_dir / 'checkpoints' / f'epoch_{epoch+1}.pt')
            )
    
    # Final test
    test_metrics = validate(model, test_loader, cfg, device)
    print(f"\nTest Loss:   {test_metrics['val_loss']:.6f}")
    print(f"Test MSE:    {test_metrics['val_rollout_mse']:.6f}")
    print(f"Test Drift:  {test_metrics['val_energy_drift']:.6f}")
    
    save_checkpoint(
        model, optimizer, cfg['training']['epochs']-1, test_metrics['val_loss'],
        str(output_dir / 'checkpoints' / 'final.pt')
    )
    
    print("\n✓ Training complete!")

if __name__ == "__main__":
    main()

print("✅ train.py 已创建")

## Phase 2: 数据准备

In [ ]:
# 生成数据
from src.data_gen import generate_burgers_1d
import numpy as np
from datetime import datetime

DATA_DIR = '/content/drive/MyDrive/srpsi-colab-baseline/data'
os.makedirs(DATA_DIR, exist_ok=True)

DATA_PATH = os.path.join(DATA_DIR, 'burgers_1d_v1.npy')

if not os.path.exists(DATA_PATH):
    print("📊 生成 Burgers 1D 数据...")
    
    data = generate_burgers_1d(
        num_samples=4800,
        total_steps=48,
        nx=128,
        dt=0.01,
        dx=0.01,
        nu=0.01,
        seed=42
    )
    
    np.save(DATA_PATH, data)
    print(f"\n✅ 数据已保存: {DATA_PATH}")
    print(f"   Shape: {data.shape}")
else:
    print(f"✅ 数据已存在: {DATA_PATH}")
    data = np.load(DATA_PATH)
    print(f"   Shape: {data.shape}")

## Phase 3: 训练

In [ ]:
# 配置训练
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = f'/content/drive/MyDrive/srpsi-colab-baseline/runs/v1_baseline_{timestamp}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 输出目录: {OUTPUT_DIR}")
print(f"📊 数据路径: {DATA_PATH}")
print(f"\n🚀 准备开始训练 (80 epochs, 预计 60-90 分钟)...")

In [ ]:
# 执行训练
!python train.py \
    --config config/burgers.yaml \
    --data {DATA_PATH} \
    --epochs 80 \
    --output {OUTPUT_DIR}

## Phase 4: 结果分析

In [ ]:
# 加载并测试模型
import torch
from train import create_model, load_config
from src.utils import get_device
from src.datasets import create_dataloaders

cfg = load_config('config/burgers.yaml')
device = get_device('cuda')

model = create_model(cfg, device)
checkpoint_path = f"{OUTPUT_DIR}/checkpoints/final.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

train_loader, val_loader, test_loader = create_dataloaders(
    data_path=DATA_PATH,
    tin=16, tout=32, batch_size=32,
    num_train=4000, num_val=400, num_test=400,
    num_workers=0
)

from train import validate
test_metrics = validate(model, test_loader, cfg, device)

print("\n" + "="*60)
print(" "*15 + "FINAL RESULTS")
print("="*60)
print(f"Test Loss:      {test_metrics['val_loss']:.6f}")
print(f"Test MSE:       {test_metrics['val_rollout_mse']:.6f}")
print(f"Test Drift:     {test_metrics['val_energy_drift']:.6f}")
print()

baseline_drift = 10.88
if test_metrics['val_energy_drift'] < baseline_drift:
    print(f"   ✅ Energy Drift 改善: {baseline_drift:.2f} → {test_metrics['val_energy_drift']:.2f}")
else:
    print(f"   Energy Drift: {test_metrics['val_energy_drift']:.2f} (基线: {baseline_drift:.2f})")

## Phase 5: 保存结果

In [ ]:
import json

results = {
    'model': 'SRΨ-v1.0 (Baseline)',
    'timestamp': datetime.now().isoformat(),
    'test_metrics': {
        'loss': float(test_metrics['val_loss']),
        'rollout_mse': float(test_metrics['val_rollout_mse']),
        'energy_drift': float(test_metrics['val_energy_drift'])
    },
    'baseline_comparison': {
        'v1_energy_drift': 10.88,
        'diff_vs_baseline': f"{((test_metrics['val_energy_drift'] - 10.88) / 10.88 * 100):.1f}%"
    }
}

results_path = f"{OUTPUT_DIR}/test_results.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"💾 结果已保存: {results_path}")
print()
print("="*60)
print(" "*15 + "TRAINING COMPLETE ✅")
print("="*60)